In [21]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


Loading Datasets (mnist, cifar10_gray, madelon)

In [22]:
class DatasetLoader:
    """
    Handles loading, preprocessing, and splitting of datasets.
    Designed for feature selection experiments.
    """

    def __init__(self, dataset_name: str):
        self.dataset_name = dataset_name
        self.X = None
        self.y = None
        self.feature_names = None

    def load(self):
        print(f"\n📦 Loading dataset: {self.dataset_name} ...")

        if self.dataset_name == "mnist":
            data = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
            self.X = data.data.astype(np.float32)
            self.y = data.target.astype(int)
            self.feature_names = [f"pixel_{i}" for i in range(self.X.shape[1])]

        elif self.dataset_name == "cifar10_gray":
            from keras.datasets import cifar10
            import cv2
            (X_train, y_train), (X_test, y_test) = cifar10.load_data()
            # Convert to grayscale and flatten → 1024 features
            X_all = np.concatenate([X_train, X_test], axis=0)
            y_all = np.concatenate([y_train.flatten(), y_test.flatten()], axis=0)
            X_gray = np.array([
                cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).flatten()
                for img in X_all
            ], dtype=np.float32)
            self.X = X_gray
            self.y = y_all
            self.feature_names = [f"pixel_{i}" for i in range(self.X.shape[1])]

        elif self.dataset_name == "madelon":
            data = fetch_openml('madelon', version=1, as_frame=False, parser='auto')
            self.X = data.data.astype(np.float32)
            self.y = (data.target.astype(int) > 0).astype(int)  # binary
            self.feature_names = [f"feat_{i}" for i in range(self.X.shape[1])]

        else:
            raise ValueError(f"Unknown dataset: {self.dataset_name}")

        print(f"   Shape     : {self.X.shape}")
        print(f"   Classes   : {np.unique(self.y)}")
        print(f"   Features  : {self.X.shape[1]}")
        return self

    def preprocess(self, sample_size=None):
        """Scale features to [0,1]. Optionally subsample for speed."""
        if sample_size and sample_size < len(self.X):
            idx = np.random.choice(len(self.X), sample_size, replace=False)
            self.X = self.X[idx]
            self.y = self.y[idx]
            print(f"   Subsampled to {sample_size} samples")

        scaler = MinMaxScaler()
        self.X = scaler.fit_transform(self.X)
        print(f"   ✅ Preprocessing done. Feature range: [{self.X.min():.2f}, {self.X.max():.2f}]")
        return self

    def split(self, test_size=0.2, random_state=42):
        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y, test_size=test_size, random_state=random_state, stratify=self.y
        )
        print(f"   Train: {X_train.shape}, Test: {X_test.shape}")
        return X_train, X_test, y_train, y_test

Naive Bayes model for evaluation

In [23]:
class NaiveBayesEvaluator:
    """
    Wraps GaussianNaiveBayes for use as the ML evaluator
    in the feature selection pipeline.
    This will be called by the Fitness Function
    """

    def __init__(self):
        self.model = GaussianNB()

    def evaluate(self, X_train, X_test, y_train, y_test, feature_mask=None):
        """
        Evaluate accuracy using a binary feature mask.
        feature_mask: array of 0s and 1s (e.g., [1,0,1,1,...])
                      If None, uses ALL features (baseline).
        Returns: accuracy (float), time_taken (float)
        """
        if feature_mask is not None:
            selected = np.where(np.array(feature_mask) == 1)[0]
            if len(selected) == 0:
                return 0.0, 0.0
            X_tr = X_train[:, selected]
            X_te = X_test[:, selected]
        else:
            X_tr, X_te = X_train, X_test

        start = time.time()
        self.model.fit(X_tr, y_train)
        preds = self.model.predict(X_te)
        elapsed = time.time() - start

        acc = accuracy_score(y_test, preds)
        return acc, elapsed

Loading and Pre-processing Datasets

In [24]:
# ─────────────────────────────────────────────
# 1. DATASET LOADER
# ─────────────────────────────────────────────
class DatasetLoader:
    def __init__(self, dataset_name: str):
        self.dataset_name = dataset_name
        self.X = None
        self.y = None

    def load(self):
        print(f"\n📦 Loading dataset: {self.dataset_name} ...")

        if self.dataset_name == "mnist":
          from keras.datasets import mnist
          (X_train, y_train), (X_test, y_test) = mnist.load_data()
          X_all = np.concatenate([X_train, X_test], axis=0)
          y_all = np.concatenate([y_train, y_test], axis=0)
          # Flatten 28x28 images into 784 features
          self.X = X_all.reshape(X_all.shape[0], -1).astype(np.float32)
          self.y = y_all.astype(int)

        elif self.dataset_name == "cifar10_gray":
            from keras.datasets import cifar10
            import cv2
            (X_train, y_train), (X_test, y_test) = cifar10.load_data()
            X_all = np.concatenate([X_train, X_test], axis=0)
            y_all = np.concatenate([y_train.flatten(), y_test.flatten()], axis=0)
            self.X = np.array([
                cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).flatten()
                for img in X_all
            ], dtype=np.float32)
            self.y = y_all

        elif self.dataset_name == "madelon":
            from sklearn.datasets import make_classification
            X, y = make_classification(
                n_samples=2600,
                n_features=500,
                n_informative=20,
                n_redundant=480,
                n_classes=2,
                random_state=42
            )
            self.X = X.astype(np.float32)
            self.y = y.astype(int)

        else:
            raise ValueError(f"Unknown dataset: {self.dataset_name}")

        print(f"   Shape     : {self.X.shape}")
        print(f"   Classes   : {np.unique(self.y)}")
        print(f"   Features  : {self.X.shape[1]}")
        return self

    def preprocess(self, sample_size=None):
        if sample_size and sample_size < len(self.X):
            np.random.seed(42)
            idx = np.random.choice(len(self.X), sample_size, replace=False)
            self.X = self.X[idx]
            self.y = self.y[idx]
            print(f"   Subsampled to {sample_size} samples")

        scaler = MinMaxScaler()
        self.X = scaler.fit_transform(self.X)
        print(f"   ✅ Preprocessing done. Feature range: [{self.X.min():.2f}, {self.X.max():.2f}]")
        return self

    def split(self, test_size=0.2, random_state=42):
        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y,
            test_size=test_size,
            random_state=random_state,
            stratify=self.y
        )
        print(f"   Train: {X_train.shape}, Test: {X_test.shape}")
        return X_train, X_test, y_train, y_test



Naive Bayes Evaluator

In [25]:
# ─────────────────────────────────────────────
# 2. NAIVE BAYES EVALUATOR
# ─────────────────────────────────────────────
class NaiveBayesEvaluator:
    def __init__(self):
        self.model = GaussianNB()

    def evaluate(self, X_train, X_test, y_train, y_test, feature_mask=None):
        if feature_mask is not None:
            selected = np.where(np.array(feature_mask) == 1)[0]
            if len(selected) == 0:
                return 0.0, 0.0
            X_tr = X_train[:, selected]
            X_te = X_test[:, selected]
        else:
            X_tr, X_te = X_train, X_test

        start = time.time()
        self.model.fit(X_tr, y_train)
        preds = self.model.predict(X_te)
        elapsed = time.time() - start

        acc = accuracy_score(y_test, preds)
        return acc, elapsed


# ─────────────────────────────────────────────
# 3. BASELINE RUNNER
# ─────────────────────────────────────────────
def run_baseline(dataset_name, sample_size=5000):
    print(f"\n{'='*50}")
    print(f"  BASELINE: {dataset_name.upper()}")
    print(f"{'='*50}")

    loader = DatasetLoader(dataset_name)
    loader.load().preprocess(sample_size=sample_size)
    X_train, X_test, y_train, y_test = loader.split()

    evaluator = NaiveBayesEvaluator()
    acc, t = evaluator.evaluate(X_train, X_test, y_train, y_test, feature_mask=None)

    # Sanity check — print both train and test accuracy for madelon
    if dataset_name == "madelon":
        train_acc = accuracy_score(y_train, evaluator.model.predict(X_train))
        print(f"\n🔍 Sanity Check:")
        print(f"   Train Accuracy : {train_acc * 100:.2f}%")
        print(f"   Test  Accuracy : {acc * 100:.2f}%")

    print(f"\n📊 Results (ALL {X_train.shape[1]} features):")
    print(f"   Accuracy : {acc * 100:.2f}%")
    print(f"   Time     : {t:.4f}s")

    return {
        "dataset": dataset_name,
        "n_features": X_train.shape[1],
        "accuracy_full": acc,
        "time_full": t,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }


# ─────────────────────────────────────────────
# 4. RUN ALL 3 DATASETS
# ─────────────────────────────────────────────
results = {}
results["mnist"]        = run_baseline("mnist",        sample_size=5000)
results["madelon"]      = run_baseline("madelon",      sample_size=None)
results["cifar10_gray"] = run_baseline("cifar10_gray", sample_size=5000)



  BASELINE: MNIST

📦 Loading dataset: mnist ...
   Shape     : (70000, 784)
   Classes   : [0 1 2 3 4 5 6 7 8 9]
   Features  : 784
   Subsampled to 5000 samples
   ✅ Preprocessing done. Feature range: [0.00, 1.00]
   Train: (4000, 784), Test: (1000, 784)

📊 Results (ALL 784 features):
   Accuracy : 58.70%
   Time     : 0.1297s

  BASELINE: MADELON

📦 Loading dataset: madelon ...
   Shape     : (2600, 500)
   Classes   : [0 1]
   Features  : 500
   ✅ Preprocessing done. Feature range: [0.00, 1.00]
   Train: (2080, 500), Test: (520, 500)

🔍 Sanity Check:
   Train Accuracy : 86.54%
   Test  Accuracy : 84.04%

📊 Results (ALL 500 features):
   Accuracy : 84.04%
   Time     : 0.0151s

  BASELINE: CIFAR10_GRAY

📦 Loading dataset: cifar10_gray ...
   Shape     : (60000, 1024)
   Classes   : [0 1 2 3 4 5 6 7 8 9]
   Features  : 1024
   Subsampled to 5000 samples
   ✅ Preprocessing done. Feature range: [0.00, 1.00]
   Train: (4000, 1024), Test: (1000, 1024)

📊 Results (ALL 1024 features):
   A

Summary Table

In [26]:
summary = pd.DataFrame([{
    "Dataset":           r["dataset"],
    "Num Features":      r["n_features"],
    "Full Accuracy (%)": round(r["accuracy_full"] * 100, 2),
    "Time (s)":          round(r["time_full"], 4)
} for r in results.values()])

print("\n📋 BASELINE SUMMARY TABLE")
print(summary.to_string(index=False))

summary.to_csv("baseline_results.csv", index=False)
print("\n✅ Saved to baseline_results.csv")


📋 BASELINE SUMMARY TABLE
     Dataset  Num Features  Full Accuracy (%)  Time (s)
       mnist           784              58.70    0.1297
     madelon           500              84.04    0.0151
cifar10_gray          1024              22.70    0.1528

✅ Saved to baseline_results.csv
